# Week 10, Day 2: Differential equations and the leaky integrator

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w010/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Exact solution against Euler steps

Write `V_exact(t)` for a step input from rest, and `euler(dt, n_steps)` that applies $V \leftarrow V + \frac{\Delta t}{\tau}(-V + RI)$. Compare the voltage at 50 ms for time steps of 1 ms and 0.1 ms, storing the two absolute errors in `err`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
tau, RI = 0.020, 30.0                 # 20 ms; input worth 30 mV
V_inf = RI

def V_exact(t):
    return ...

def euler(dt, n_steps):
    V = np.zeros(n_steps + 1)
    for k in range(n_steps):
        V[k + 1] = ...
    return V

err = [abs(euler(dt, int(round(0.05 / dt)))[-1] - V_exact(0.05)) for dt in [0.001, 0.0001]]
t = np.linspace(0, 0.1, 200)
plt.figure(figsize=(5, 3)); plt.plot(t * 1000, V_exact(t), label="exact"); plt.plot(np.arange(101) * 1, euler(0.001, 100), ".", ms=3, label="Euler, 1 ms")
plt.xlabel("time (ms)"); plt.ylabel("V (mV)"); plt.legend(frameon=False); plt.show()
print("errors at 50 ms:", np.round(err, 5))

In [ ]:
import numpy as np
def test_exact():
    "V_exact follows the exponential approach to V_inf"
    assert np.isclose(V_exact(0.0), 0) and np.isclose(V_exact(tau), V_inf * (1 - np.exp(-1)))

def test_euler():
    "euler returns n_steps + 1 values starting at 0"
    v = euler(0.001, 50)
    assert len(v) == 51 and v[0] == 0

def test_order():
    "Euler's error at 50 ms shrinks about tenfold when dt shrinks tenfold"
    assert 8 < err[0] / err[1] < 12

test_exact()
test_euler()
test_order()
print("All checks passed")

### Exercise: Convolution reproduces the simulation

Drive the membrane with a noisy current (the cell sets it up). Simulate it with Euler steps (`V_sim`), then compute the same response by convolving the input with the exponential kernel: `V_conv = np.convolve(RI_t, kernel)[:n] * dt`. Plot both. Which features of the input survive the filtering?

In [ ]:
dt, n = 0.0001, 3000                     # 0.1 ms steps, 300 ms
rng = np.random.default_rng(1907)
RI_t = 15 + 40 * rng.normal(size=n) * np.sqrt(0.001 / dt) * 0.3 + 10 * np.sin(2 * np.pi * 8 * np.arange(n) * dt)

V_sim = np.zeros(n)
for k in range(n - 1):
    V_sim[k + 1] = ...

kernel = ...                             # (1 / tau) exp(-t / tau) for t = 0, dt, 2 dt, ... up to 10 tau
V_conv = np.convolve(RI_t, kernel)[:n] * dt

t = np.arange(n) * dt * 1000
plt.figure(figsize=(6, 3)); plt.plot(t, RI_t, color="0.8", lw=0.5, label="input"); plt.plot(t, V_sim, label="Euler"); plt.plot(t, V_conv, "--", label="convolution")
plt.ylim(-40, 80); plt.xlabel("time (ms)"); plt.ylabel("mV"); plt.legend(frameon=False, ncol=3); plt.show()
print("largest difference:", np.max(np.abs(V_conv - V_sim)).round(3), "mV")

In [ ]:
import numpy as np
def test_kernel():
    "kernel is (1 / tau) exp(-t / tau) sampled every dt, and integrates to about 1"
    assert np.isclose(kernel[0], 1 / tau) and abs(np.sum(kernel) * dt - 1) < 0.01

def test_match():
    "the convolution matches the Euler simulation to within 1 mV"
    assert np.max(np.abs(V_conv - V_sim)) < 1.0

test_kernel()
test_match()
print("All checks passed")

### Exercise: Simulate and check the f-I curve

Write `f_theory(RI)` from the proof (0 at or below threshold). Then simulate the integrate-and-fire neuron for 2 s with each constant input in `inputs`, using exact updates between spikes, a threshold of 15 mV, reset to 0 and a 2 ms refractory period, and store the firing rates in `rates_sim`. Plot both curves.

In [ ]:
theta, t_ref = 15.0, 0.002
def f_theory(RIv):
    if RIv <= theta:
        return 0.0
    return ...

def simulate_lif(RIv, T=2.0, dt=0.00005):
    V, spikes, refractory = 0.0, 0, 0.0
    decay = np.exp(-dt / tau)
    for _ in range(int(T / dt)):
        if refractory > 0:
            refractory -= dt
            continue
        V = RIv + (V - RIv) * decay          # exact update for a constant input over one step
        if V >= theta:
            ...
    return spikes / T

inputs = [10.0, 18.0, 22.0, 30.0, 45.0, 70.0]
rates_sim = [simulate_lif(v) for v in inputs]
grid = np.linspace(5, 75, 300)
plt.figure(figsize=(5, 3)); plt.plot(grid, [f_theory(v) for v in grid], label="formula"); plt.plot(inputs, rates_sim, "o", label="simulation")
plt.xlabel("input RI (mV)"); plt.ylabel("firing rate (Hz)"); plt.legend(frameon=False); plt.show()
print([round(r, 1) for r in rates_sim], " formula at 30 mV:", round(f_theory(30.0), 1), "Hz")

In [ ]:
import numpy as np
def test_rates():
    "rates_sim has one simulated firing rate per input"
    assert len(rates_sim) == len(inputs)

def test_silent():
    "inputs below threshold produce no spikes"
    assert rates_sim[0] == 0 and f_theory(10.0) == 0

def test_match():
    "simulated rates match the formula within 3% above threshold"
    for RIv, r in zip(inputs, rates_sim):
        if RIv > 16:
            assert abs(r - f_theory(RIv)) / f_theory(RIv) < 0.03, (RIv, r, f_theory(RIv))

test_rates()
test_silent()
test_match()
print("All checks passed")